In [1]:
!find /kaggle/input -maxdepth 4

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/mdtaufiquehussain
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code/model
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code/tests
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code/hindi
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code/.gitignore
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code/collection
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code/README.md
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code/report
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code/run_phase1.py
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code/requirements.txt
/kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code/nepali
/kaggle/input/datasets/mdtaufiquehussain/lma-phase-hindi-data
/kaggle/input/datasets/mdtaufiquehussain/lma-phase-hindi-data/LMA-Phase1-Hindi


In [2]:
import shutil
from pathlib import Path

def find_dataset_root(slug: str) -> Path:
    for candidate in Path("/kaggle/input").glob(f"**/{slug}"):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(f"could not find dataset '{slug}' under /kaggle/input")

CODE_SRC = find_dataset_root("lma-phase2-code")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(CODE_SRC, REPO)
print(f"copied code from {CODE_SRC} -> {REPO}")
print(list(REPO.iterdir()))

copied code from /kaggle/input/datasets/mdtaufiquehussain/lma-phase2-code -> /kaggle/working/repo
[PosixPath('/kaggle/working/repo/collection'), PosixPath('/kaggle/working/repo/hindi'), PosixPath('/kaggle/working/repo/requirements.txt'), PosixPath('/kaggle/working/repo/model'), PosixPath('/kaggle/working/repo/tests'), PosixPath('/kaggle/working/repo/report'), PosixPath('/kaggle/working/repo/run_phase1.py'), PosixPath('/kaggle/working/repo/nepali'), PosixPath('/kaggle/working/repo/.gitignore'), PosixPath('/kaggle/working/repo/README.md')]


In [3]:
import subprocess
subprocess.run(["pip", "install", "-q", "sentencepiece"], check=True)

CompletedProcess(args=['pip', 'install', '-q', 'sentencepiece'], returncode=0)

In [4]:
CHECKPOINT_INPUT_DIR = None
CHECKPOINT_OUTPUT_DIR = Path("/kaggle/working/checkpoints/hindi")

if CHECKPOINT_INPUT_DIR is not None and Path(CHECKPOINT_INPUT_DIR).exists():
    CHECKPOINT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for f in Path(CHECKPOINT_INPUT_DIR).glob("*.pt"):
        shutil.copy(f, CHECKPOINT_OUTPUT_DIR / f.name)
    print(f"copied {list(CHECKPOINT_OUTPUT_DIR.glob('*.pt'))} for resume")
else:
    print("no checkpoint input configured -- starting fresh")

no checkpoint input configured -- starting fresh


In [5]:
import sys
sys.path.insert(0, "/kaggle/working/repo")
from model.config import ModelConfig
from model.train_loop import TrainConfig, train
from hindi.train.train import prepare_binaries

DATASET_ROOT = find_dataset_root("lma-phase-hindi-data")
SPLITS_DIR = next(DATASET_ROOT.glob("**/splits"))
print(f"found splits dir: {SPLITS_DIR}")
TOKENIZER_PATH = REPO / "hindi" / "tokenizer" / "hindi_tokenizer.model"
BIN_DIR = Path("/kaggle/working/tokenized/hindi")

model_cfg = ModelConfig.load(REPO / "hindi" / "configs" / "model.yaml")
train_cfg = TrainConfig.load(REPO / "hindi" / "configs" / "train.yaml")
print(f"model: {model_cfg}")
print(f"train: {train_cfg}")

train_bin, val_bin = prepare_binaries(SPLITS_DIR, TOKENIZER_PATH, BIN_DIR)

train(
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    train_bin=train_bin,
    val_bin=val_bin,
    checkpoint_dir=CHECKPOINT_OUTPUT_DIR,
    resume=True,
)

found splits dir: /kaggle/input/datasets/mdtaufiquehussain/lma-phase-hindi-data/LMA-Phase1-Hindi/splits
model: ModelConfig(vocab_size=9000, d_model=384, n_layers=13, n_heads=6, d_ff=1536, max_seq_len=256, dropout=0.1)
train: TrainConfig(lr=0.0003, weight_decay=0.1, beta1=0.9, beta2=0.95, grad_clip=1.0, warmup_steps=1000, max_steps=61000, batch_size=32, eval_every=500, eval_batches=50, checkpoint_every=500, log_every=50)
[hindi] tokenizing train split -> /kaggle/working/tokenized/hindi/train.bin
[hindi] train: 683,611,144 tokens
[hindi] tokenizing val split -> /kaggle/working/tokenized/hindi/val.bin
[hindi] val: 6,938,359 tokens
starting fresh: 26,524,800 trainable params
step 50/61000 | train_loss 8.5920 | lr 1.50e-05 | 27s elapsed
step 100/61000 | train_loss 8.1936 | lr 3.00e-05 | 53s elapsed
step 150/61000 | train_loss 7.6718 | lr 4.50e-05 | 80s elapsed
step 200/61000 | train_loss 7.0970 | lr 6.00e-05 | 108s elapsed
step 250/61000 | train_loss 6.8612 | lr 7.50e-05 | 137s elapsed
step

In [6]:
for f in CHECKPOINT_OUTPUT_DIR.glob("*.pt"):
    print(f, f.stat().st_size / 1e6, "MB")

/kaggle/working/checkpoints/hindi/best.pt 318.488583 MB
/kaggle/working/checkpoints/hindi/latest.pt 318.490443 MB
